# 11. 동시성과 비동기 처리 예제

## Goal

- 동시 작업 수를 제한합니다.
- 일부 실패를 결과 목록에 보존합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

합성 I/O 작업만 사용합니다. 프로세스 풀과 외부 HTTP 요청은 실행하지 않습니다.


## Steps

### 제한된 스레드와 비동기 작업

작업별 결과를 구조화하고 입력 순서와 결과 식별자를 유지합니다.


In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import asyncio
import threading
import time


def simulated_io(item):
    time.sleep(0.01)
    if item == "bad":
        raise OSError("합성 I/O 오류")
    return item.upper()


def run_threaded(items, workers=2):
    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(simulated_io, item): item for item in items}
        for future in as_completed(futures):
            item = futures[future]
            try:
                results.append({"item": item, "status": "ok", "value": future.result()})
            except OSError as error:
                results.append({"item": item, "status": "error", "error": str(error)})
    return sorted(results, key=lambda row: items.index(row["item"]))


thread_results = run_threaded(["one", "bad", "two"])
for row in thread_results:
    print(row)


async def bounded_double(values, limit=2):
    semaphore = asyncio.Semaphore(limit)
    async def one(value):
        async with semaphore:
            await asyncio.sleep(0.005)
            return value * 2
    return await asyncio.gather(*(one(value) for value in values))


def run_async_in_worker(coroutine):
    box = []
    worker = threading.Thread(target=lambda: box.append(asyncio.run(coroutine)))
    worker.start()
    worker.join()
    return box[0]


async_results = run_async_in_worker(bounded_double([1, 2, 3]))
print("비동기 결과:", async_results)


{'item': 'one', 'status': 'ok', 'value': 'ONE'}
{'item': 'bad', 'status': 'error', 'error': '합성 I/O 오류'}
{'item': 'two', 'status': 'ok', 'value': 'TWO'}
비동기 결과: [2, 4, 6]


## Checks

오류 보존과 비동기 결과를 확인합니다.


In [2]:
assert [row["status"] for row in thread_results] == ["ok", "error", "ok"]
assert async_results == [2, 4, 6]
assert thread_results[1]["item"] == "bad"
print("동시성 결과 검사 통과")


동시성 결과 검사 통과


## Next Steps

실제 HTTP 작업에는 전체 타임아웃·요청 속도·취소 후 정리 정책을 추가합니다.
